In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

In [2]:
# =========================
# PATHS
# =========================
TRAIN_PATH_1 = "segment_alerts_all_airports_train.csv"
#TRAIN_PATH_2 = "dataset_set.csv"   # mets None si tu n'en as pas besoin
TEST_PATH = "dataset_set.csv"             # adapte au vrai nom du fichier test

# =========================
# LOAD
# =========================
df_train = pd.read_csv(TRAIN_PATH_1)

df_test = pd.read_csv(TEST_PATH)

for df_ in [df_train, df_test]:
    if "Unnamed: 0" in df_.columns:
        df_.drop(columns=["Unnamed: 0"], inplace=True, errors="ignore")
    df_["date"] = pd.to_datetime(df_["date"], utc=True)
    df_["icloud"] = df_["icloud"].astype(bool)

print("Train shape:", df_train.shape)
print("Test shape :", df_test.shape)

print("\nTrain years:")
print(df_train["date"].dt.year.value_counts().sort_index())

print("\nTest years:")
print(df_test["date"].dt.year.value_counts().sort_index())

print("\nTrain columns:")
print(sorted(df_train.columns.tolist()))

print("\nTest columns:")
print(sorted(df_test.columns.tolist()))

NameError: name 'TRAIN_PATH_2' is not defined

In [ ]:
def month_to_season(m):
    if m in [12, 1, 2]:
        return "winter"
    elif m in [3, 4, 5]:
        return "spring"
    elif m in [6, 7, 8]:
        return "summer"
    else:
        return "autumn"


def add_base_columns(df, strong_cg_threshold=None):
    df = df.copy()

    df["is_cg"] = (~df["icloud"]).astype("int8")
    df["is_ic"] = df["icloud"].astype("int8")

    df["is_0_3"] = (df["dist"] < 3).astype("int8")
    df["is_3_10"] = ((df["dist"] >= 3) & (df["dist"] < 10)).astype("int8")
    df["is_10_20"] = ((df["dist"] >= 10) & (df["dist"] < 20)).astype("int8")
    df["is_20_30"] = ((df["dist"] >= 20) & (df["dist"] < 30)).astype("int8")

    df["amp_abs"] = df["amplitude"].abs()

    if strong_cg_threshold is None:
        strong_cg_threshold = df.loc[df["is_cg"] == 1, "amp_abs"].quantile(0.90)

    df["is_strong_cg"] = ((df["is_cg"] == 1) & (df["amp_abs"] >= strong_cg_threshold)).astype("int8")

    df["hour"] = df["date"].dt.hour
    df["month"] = df["date"].dt.month
    df["dayofyear"] = df["date"].dt.dayofyear
    df["weekday"] = df["date"].dt.weekday

    df["season"] = df["month"].apply(month_to_season)
    df["is_summer"] = (df["season"] == "summer").astype("int8")
    df["is_peak_hour"] = df["hour"].between(12, 18).astype("int8")

    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

    az_rad = np.deg2rad(df["azimuth"])
    df["azimuth_sin"] = np.sin(az_rad)
    df["azimuth_cos"] = np.cos(az_rad)

    az_bins = [0, 45, 90, 135, 180, 225, 270, 315, 360]
    az_labels = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]
    df["az_sector"] = pd.cut(
        df["azimuth"].mod(360),
        bins=az_bins,
        labels=az_labels,
        include_lowest=True,
        right=False
    )

    df["cg_amp_abs"] = np.where(df["is_cg"] == 1, df["amp_abs"], np.nan)

    return df, strong_cg_threshold


def build_timestamp_features(df_all):
    t0 = time.time()

    base = (
        df_all.groupby(["airport", "date"], as_index=False)
              .agg(
                  n_total=("lightning_id", "size"),
                  n_cg=("is_cg", "sum"),
                  n_ic=("is_ic", "sum"),
                  n_strong_cg=("is_strong_cg", "sum"),

                  n_0_3=("is_0_3", "sum"),
                  n_3_10=("is_3_10", "sum"),
                  n_10_20=("is_10_20", "sum"),
                  n_20_30=("is_20_30", "sum"),

                  dist_min=("dist", "min"),
                  dist_mean=("dist", "mean"),
                  dist_max=("dist", "max"),

                  amp_abs_mean=("amp_abs", "mean"),
                  amp_abs_max=("amp_abs", "max"),
                  amp_abs_sum=("amp_abs", "sum"),

                  cg_amp_abs_mean=("cg_amp_abs", "mean"),
                  cg_amp_abs_max=("cg_amp_abs", "max"),

                  maxis_mean=("maxis", "mean"),
                  azimuth_mean=("azimuth", "mean"),
                  azimuth_sin_mean=("azimuth_sin", "mean"),
                  azimuth_cos_mean=("azimuth_cos", "mean"),

                  hour=("hour", "first"),
                  month=("month", "first"),
                  dayofyear=("dayofyear", "first"),
                  weekday=("weekday", "first"),
                  season=("season", "first"),
                  is_summer=("is_summer", "first"),
                  is_peak_hour=("is_peak_hour", "first"),
                  hour_sin=("hour_sin", "first"),
                  hour_cos=("hour_cos", "first"),
                  month_sin=("month_sin", "first"),
                  month_cos=("month_cos", "first"),
              )
              .sort_values(["airport", "date"])
              .reset_index(drop=True)
    )

    base["cg_amp_abs_mean"] = base["cg_amp_abs_mean"].fillna(0.0)
    base["cg_amp_abs_max"] = base["cg_amp_abs_max"].fillna(0.0)

    base["cg_ratio_now"] = base["n_cg"] / (base["n_total"] + 1e-6)
    base["strong_cg_ratio_now"] = base["n_strong_cg"] / (base["n_cg"] + 1e-6)

    sector_counts = (
        df_all.groupby(["airport", "date", "az_sector"], observed=False)
              .size()
              .unstack(fill_value=0)
              .reset_index()
    )
    sector_cols = [c for c in sector_counts.columns if c not in ["airport", "date"]]
    sector_counts = sector_counts.rename(columns={c: f"sector_{c}_count" for c in sector_cols})

    base = base.merge(sector_counts, on=["airport", "date"], how="left")
    for c in base.columns:
        if c.startswith("sector_") and c.endswith("_count"):
            base[c] = base[c].fillna(0)

    print("build_timestamp_features:", round(time.time() - t0, 2), "s")
    return base


def add_since_features(base):
    t0 = time.time()
    parts = []

    for airport, g in base.groupby("airport"):
        g = g.sort_values("date").copy()

        g["delta_prev_min"] = g["date"].diff().dt.total_seconds().div(60).fillna(9999)

        def since_last(col_name, new_name):
            last_date = g["date"].where(g[col_name] > 0).ffill().shift()
            g[new_name] = (g["date"] - last_date).dt.total_seconds().div(60).fillna(9999)

        since_last("n_0_3", "since_last_0_3_min")
        since_last("n_3_10", "since_last_3_10_min")
        since_last("n_10_20", "since_last_10_20_min")
        since_last("n_20_30", "since_last_20_30_min")
        since_last("n_cg", "since_last_cg_min")
        since_last("n_strong_cg", "since_last_strong_cg_min")

        parts.append(g)

    out = pd.concat(parts, ignore_index=True)
    print("add_since_features:", round(time.time() - t0, 2), "s")
    return out


def add_rolling_features(base_time):
    t0 = time.time()

    windows = ["5min", "10min", "30min"]

    count_cols = [
        "n_total", "n_cg", "n_ic", "n_strong_cg",
        "n_0_3", "n_3_10", "n_10_20", "n_20_30"
    ]
    sector_cols = [c for c in base_time.columns if c.startswith("sector_") and c.endswith("_count")]
    count_cols_all = count_cols + sector_cols

    mean_cols = [
        "dist_min", "dist_mean", "dist_max",
        "amp_abs_mean", "amp_abs_max", "amp_abs_sum",
        "cg_amp_abs_mean", "cg_amp_abs_max",
        "cg_ratio_now", "strong_cg_ratio_now"
    ]

    parts = []

    for airport, g in base_time.groupby("airport"):
        g = g.sort_values("date").copy().set_index("date")
        out = g.copy()

        for w in windows:
            roll_sum = g[count_cols_all].rolling(w, closed="both").sum()
            roll_sum.columns = [f"{c}__sum_{w}" for c in roll_sum.columns]

            roll_mean = g[mean_cols].rolling(w, closed="both").mean()
            roll_mean.columns = [f"{c}__mean_{w}" for c in roll_mean.columns]

            out = pd.concat([out, roll_sum, roll_mean], axis=1)

        out = out.reset_index()
        parts.append(out)

    out = pd.concat(parts, ignore_index=True)
    print("add_rolling_features:", round(time.time() - t0, 2), "s")
    return out


def add_derived_features(features_all):
    eps = 1e-6
    df = features_all.copy()

    df["trend_total_5_vs_10"] = df["n_total__sum_5min"] / (df["n_total__sum_10min"] + eps)
    df["trend_0_3_5_vs_10"] = df["n_0_3__sum_5min"] / (df["n_0_3__sum_10min"] + eps)
    df["trend_10_20_5_vs_10"] = df["n_10_20__sum_5min"] / (df["n_10_20__sum_10min"] + eps)
    df["trend_20_30_5_vs_10"] = df["n_20_30__sum_5min"] / (df["n_20_30__sum_10min"] + eps)
    df["trend_cg_5_vs_10"] = df["n_cg__sum_5min"] / (df["n_cg__sum_10min"] + eps)

    df["far_vs_near_5min"] = (df["n_20_30__sum_5min"] + 1) / (df["n_0_3__sum_5min"] + df["n_3_10__sum_5min"] + 1)
    df["far_vs_near_10min"] = (df["n_20_30__sum_10min"] + 1) / (df["n_0_3__sum_10min"] + df["n_3_10__sum_10min"] + 1)
    df["far_vs_mid_10min"] = (df["n_20_30__sum_10min"] + 1) / (df["n_10_20__sum_10min"] + 1)
    df["mid_vs_near_10min"] = (df["n_10_20__sum_10min"] + 1) / (df["n_0_3__sum_10min"] + df["n_3_10__sum_10min"] + 1)

    df["dist_range_now"] = df["dist_max"] - df["dist_min"]
    df["dist_range_10min"] = df["dist_max__mean_10min"] - df["dist_min__mean_10min"]
    df["dist_drift_5_10"] = df["dist_mean__mean_5min"] - df["dist_mean__mean_10min"]

    df["severity_now"] = (
        0.4 * np.tanh(df["amp_abs_mean"] / 20.0) +
        0.4 * df["cg_ratio_now"] +
        0.2 * (1 - np.clip(df["dist_mean"] / 30.0, 0, 1))
    )

    for w in ["5min", "10min", "30min"]:
        df[f"severity_{w}"] = (
            0.4 * np.tanh(df[f"amp_abs_mean__mean_{w}"] / 20.0) +
            0.4 * df[f"cg_ratio_now__mean_{w}"] +
            0.2 * (1 - np.clip(df[f"dist_mean__mean_{w}"] / 30.0, 0, 1))
        )

    df["strong_cg_ratio_10min"] = df["n_strong_cg__sum_10min"] / (df["n_cg__sum_10min"] + eps)
    df["strong_cg_ratio_30min"] = df["n_strong_cg__sum_30min"] / (df["n_cg__sum_30min"] + eps)

    return df


def compute_target_20km(df_train):
    labeled = df_train[df_train["airport_alert_id"].notna()].copy()

    rows = []
    for (airport, alert_id), g in labeled.groupby(["airport", "airport_alert_id"]):
        g = g.sort_values("date").copy()
        times = g[["date"]].drop_duplicates().sort_values("date").copy()
        t = times["date"].to_numpy()

        dangerous_times_20 = np.array(sorted(g.loc[g["dist"] < 20, "date"].unique()))
        idx20 = np.searchsorted(dangerous_times_20, t, side="right")

        times["safe_to_end_now_20km"] = (idx20 == len(dangerous_times_20)).astype("int8")
        times["airport"] = airport
        times["airport_alert_id"] = alert_id
        rows.append(times)

    target_df = pd.concat(rows, ignore_index=True)
    return target_df, labeled


def compute_alert_life(df_with_alerts):
    if "airport_alert_id" not in df_with_alerts.columns:
        raise ValueError("airport_alert_id manquant")

    tmp = df_with_alerts[df_with_alerts["airport_alert_id"].notna()].copy()
    tmp = tmp.sort_values(["airport", "airport_alert_id", "date"]).copy()

    life_parts = []
    for (airport, alert_id), g in tmp.groupby(["airport", "airport_alert_id"]):
        g = g.sort_values("date").copy()

        g["elapsed_since_alert_start_min"] = (g["date"] - g["date"].iloc[0]).dt.total_seconds().div(60)
        g["cum_lightnings"] = np.arange(1, len(g) + 1)
        g["cum_cg"] = g["is_cg"].cumsum()
        g["cum_ic"] = g["is_ic"].cumsum()
        g["cum_strong_cg"] = g["is_strong_cg"].cumsum()

        g["cum_avg_rate"] = g["cum_lightnings"] / (g["elapsed_since_alert_start_min"] + 1)
        g["cum_cg_rate"] = g["cum_cg"] / (g["elapsed_since_alert_start_min"] + 1)
        g["cum_strong_cg_rate"] = g["cum_strong_cg"] / (g["elapsed_since_alert_start_min"] + 1)

        g_ts = (
            g.groupby(["airport", "airport_alert_id", "date"], as_index=False)
             .agg(
                 elapsed_since_alert_start_min=("elapsed_since_alert_start_min", "max"),
                 cum_lightnings=("cum_lightnings", "max"),
                 cum_cg=("cum_cg", "max"),
                 cum_ic=("cum_ic", "max"),
                 cum_strong_cg=("cum_strong_cg", "max"),
                 cum_avg_rate=("cum_avg_rate", "max"),
                 cum_cg_rate=("cum_cg_rate", "max"),
                 cum_strong_cg_rate=("cum_strong_cg_rate", "max"),
             )
        )

        life_parts.append(g_ts)

    return pd.concat(life_parts, ignore_index=True)


def compute_airport_static(df_train):
    labeled = df_train[df_train["airport_alert_id"].notna()].copy()

    alerts_stats = (
        labeled.groupby(["airport", "airport_alert_id"], as_index=False)
               .agg(
                   alert_start=("date", "min"),
                   alert_end=("date", "max"),
                   n_lightnings_alert=("lightning_id", "count"),
                   n_cg_alert=("is_cg", "sum"),
                   n_ic_alert=("is_ic", "sum"),
                   mean_amp_alert=("amp_abs", "mean"),
               )
    )

    alerts_stats["alert_duration_min"] = (
        alerts_stats["alert_end"] - alerts_stats["alert_start"]
    ).dt.total_seconds().div(60)

    airport_static = (
        alerts_stats.groupby("airport", as_index=False)
                   .agg(
                       airport_alert_duration_mean=("alert_duration_min", "mean"),
                       airport_alert_duration_median=("alert_duration_min", "median"),
                       airport_alert_n_lightnings_mean=("n_lightnings_alert", "mean"),
                       airport_alert_n_lightnings_median=("n_lightnings_alert", "median"),
                       airport_alert_cg_mean=("n_cg_alert", "mean"),
                       airport_alert_ic_mean=("n_ic_alert", "mean"),
                       airport_alert_amp_mean=("mean_amp_alert", "mean"),
                   )
    )

    return airport_static

In [ ]:
t0 = time.time()

# même seuil strong CG partout, appris sur le train
df_train_feat, strong_cg_threshold = add_base_columns(df_train, strong_cg_threshold=None)
df_test_feat, _ = add_base_columns(df_test, strong_cg_threshold=strong_cg_threshold)

# concat pour construire les features temporelles de façon causale
df_all_feat = pd.concat([df_train_feat, df_test_feat], ignore_index=True)
df_all_feat = df_all_feat.sort_values(["airport", "date"]).reset_index(drop=True)

base = build_timestamp_features(df_all_feat)
base_time = add_since_features(base)
features_all = add_rolling_features(base_time)
features_all = add_derived_features(features_all)

print("Total feature build time:", round(time.time() - t0, 2), "s")
print(features_all.shape)
print(features_all.head())

In [ ]:
target_df, labeled_train = compute_target_20km(df_train_feat)
alert_life_train = compute_alert_life(df_train_feat)
airport_static = compute_airport_static(df_train_feat)

print("target_df:", target_df.shape)
print("alert_life_train:", alert_life_train.shape)
print("airport_static:", airport_static.shape)
print(target_df["safe_to_end_now_20km"].mean())

In [ ]:
model_df_train = (
    target_df
    .merge(features_all, on=["airport", "date"], how="left")
    .merge(alert_life_train, on=["airport", "airport_alert_id", "date"], how="left")
    .merge(airport_static, on="airport", how="left")
)

model_df_train["airport"] = model_df_train["airport"].astype("category")
if "season" in model_df_train.columns:
    model_df_train["season"] = model_df_train["season"].astype("category")

print(model_df_train.shape)
print(model_df_train.head())

In [ ]:
split_train_end = pd.Timestamp("2024-01-01", tz="UTC")
split_val_end = pd.Timestamp("2025-01-01", tz="UTC")

train_mask = model_df_train["date"] < split_train_end
val_mask = (model_df_train["date"] >= split_train_end) & (model_df_train["date"] < split_val_end)

drop_cols = ["airport_alert_id", "date", "safe_to_end_now_20km"]
feature_cols = [c for c in model_df_train.columns if c not in drop_cols]

X_train = model_df_train.loc[train_mask, feature_cols].copy()
y_train = model_df_train.loc[train_mask, "safe_to_end_now_20km"].copy()

X_val = model_df_train.loc[val_mask, feature_cols].copy()
y_val = model_df_train.loc[val_mask, "safe_to_end_now_20km"].copy()

for X in [X_train, X_val]:
    X["airport"] = X["airport"].astype("category")
    if "season" in X.columns:
        X["season"] = X["season"].astype("category")

print("Train:", X_train.shape, "positive rate =", y_train.mean())
print("Val  :", X_val.shape, "positive rate =", y_val.mean())

In [ ]:
cat_cols = ["airport"]
if "season" in X_train.columns:
    cat_cols.append("season")

params = dict(
    objective="binary",
    n_estimators=700,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=40,
    subsample=0.85,
    colsample_bytree=0.85,
    class_weight="balanced",
    random_state=42,
    verbose=-1
)

clf_20 = LGBMClassifier(**params)
clf_20.fit(X_train, y_train, categorical_feature=cat_cols)

val_proba_20 = clf_20.predict_proba(X_val)[:, 1]

print("ROC-AUC      :", roc_auc_score(y_val, val_proba_20))
print("Avg Precision:", average_precision_score(y_val, val_proba_20))
print("Brier score  :", brier_score_loss(y_val, val_proba_20))

In [ ]:
def evaluate_predictions(predictions_df, labeled_df, split_date, max_gap_minutes=30, min_dist=3):
    val_labeled_raw = labeled_df[
        (labeled_df["date"] >= split_date) &
        (labeled_df["date"] < pd.Timestamp("2025-01-01", tz="UTC"))
    ].copy()

    alerts_val = val_labeled_raw.groupby(["airport", "airport_alert_id"])
    tot_lightnings_val = len(val_labeled_raw[val_labeled_raw["dist"] < min_dist])

    thetas = [i / 20 for i in range(20)]
    rows = []

    for theta in thetas:
        pred_over_theta = predictions_df[predictions_df["confidence"] >= theta].copy()

        pred_over_theta_min = (
            pred_over_theta.groupby(["airport", "airport_alert_id"])["predicted_date_end_alert"]
            .min()
        )

        gain_seconds = 0.0
        missed_lights = 0
        n_alerts_predicted = 0

        for (airport, alert_id), end_alert_pred in pred_over_theta_min.items():
            if (airport, alert_id) not in alerts_val.groups:
                continue

            lightnings = alerts_val.get_group((airport, alert_id))
            end_alert_pred = pd.to_datetime(end_alert_pred, utc=True, errors="coerce")
            end_alert_baseline = pd.to_datetime(lightnings["date"], utc=True, errors="coerce").max()

            if pd.isna(end_alert_pred) or pd.isna(end_alert_baseline):
                continue

            end_alert_baseline = end_alert_baseline + pd.Timedelta(minutes=max_gap_minutes)

            gain_seconds += (end_alert_baseline - end_alert_pred).total_seconds()
            n_alerts_predicted += 1

            missed_lights += (
                pd.to_datetime(lightnings.loc[lightnings["dist"] < min_dist, "date"], utc=True, errors="coerce")
                > end_alert_pred
            ).sum()

        rows.append({
            "theta": theta,
            "gain_hours": gain_seconds / 3600,
            "missed_rate": missed_lights / max(tot_lightnings_val, 1),
            "missed_count": missed_lights,
            "n_alerts_predicted": n_alerts_predicted
        })

    return pd.DataFrame(rows)

In [ ]:
predictions_val_20 = model_df_train.loc[val_mask, ["airport", "airport_alert_id", "date"]].copy()
predictions_val_20["prediction_date"] = predictions_val_20["date"]
predictions_val_20["predicted_date_end_alert"] = predictions_val_20["date"]
predictions_val_20["confidence"] = val_proba_20
predictions_val_20 = predictions_val_20[
    ["airport", "airport_alert_id", "prediction_date", "predicted_date_end_alert", "confidence"]
].copy()

results_20 = evaluate_predictions(predictions_val_20, labeled_train, split_train_end, max_gap_minutes=30, min_dist=3)
print(results_20)

In [ ]:
ACCEPTABLE_RISK = 0.02

valid_rows = results_20[results_20["missed_rate"] < ACCEPTABLE_RISK].copy()

if len(valid_rows) > 0:
    best_row = valid_rows.sort_values("gain_hours", ascending=False).iloc[0]
    print("Meilleur theta sous contrainte :")
    print(best_row)
else:
    print("Aucun theta < 2% sur cette version brute.")

In [ ]:
X_full = model_df_train[feature_cols].copy()
y_full = model_df_train["safe_to_end_now_20km"].copy()

X_full["airport"] = X_full["airport"].astype("category")
if "season" in X_full.columns:
    X_full["season"] = X_full["season"].astype("category")

clf_20_final = LGBMClassifier(**params)
clf_20_final.fit(X_full, y_full, categorical_feature=cat_cols)

print("Refit full train done.")

In [ ]:
# candidats de prédiction = timestamps uniques du test
test_candidates = df_test_feat[["airport", "date"]].drop_duplicates().copy()

if "airport_alert_id" in df_test_feat.columns:
    test_candidates = (
        df_test_feat[["airport", "airport_alert_id", "date"]]
        .drop_duplicates()
        .copy()
    )
else:
    raise ValueError("Le test n'a pas airport_alert_id. Impossible de générer le fichier final attendu tel quel.")

# cycle de vie sur test si alert ids présents
alert_life_test = compute_alert_life(df_test_feat)

model_df_test = (
    test_candidates
    .merge(features_all, on=["airport", "date"], how="left")
    .merge(alert_life_test, on=["airport", "airport_alert_id", "date"], how="left")
    .merge(airport_static, on="airport", how="left")
)

model_df_test["airport"] = model_df_test["airport"].astype("category")
if "season" in model_df_test.columns:
    model_df_test["season"] = model_df_test["season"].astype("category")

print(model_df_test.shape)
print(model_df_test.head())

In [ ]:
X_test = model_df_test[feature_cols].copy()

X_test["airport"] = X_test["airport"].astype("category")
if "season" in X_test.columns:
    X_test["season"] = X_test["season"].astype("category")

test_proba_20 = clf_20_final.predict_proba(X_test)[:, 1]

print(test_proba_20[:10])
print("Nb prédictions:", len(test_proba_20))

In [ ]:
# =========================
# 1. Importance par gain
# =========================
gain_df = pd.DataFrame({
    "feature": clf_20.feature_name_,
    "importance_gain": clf_20.booster_.feature_importance(importance_type="gain")
}).sort_values("importance_gain", ascending=False)

print("Top features par gain :")
print(gain_df.head(20))

plt.figure(figsize=(10, 6))
plt.barh(gain_df["feature"].head(20)[::-1], gain_df["importance_gain"].head(20)[::-1])
plt.xlabel("Importance (gain)")
plt.title("Top 20 features - LightGBM gain")
plt.show()

# =========================
# 2. Permutation importance
# =========================
perm = permutation_importance(
    clf_20,
    X_val,
    y_val,
    scoring="average_precision",
    n_repeats=5,
    random_state=42
)

perm_df = pd.DataFrame({
    "feature": X_val.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)

print("\nTop features par permutation importance :")
print(perm_df.head(20))

plt.figure(figsize=(10, 6))
plt.barh(perm_df["feature"].head(20)[::-1], perm_df["importance_mean"].head(20)[::-1])
plt.xlabel("Importance moyenne (permutation)")
plt.title("Top 20 features - Permutation Importance")
plt.show()

In [ ]:
OUTPUT_PATH = "submission_20km_full_pipeline.csv"
submission.to_csv(OUTPUT_PATH, index=False)

print("Fichier sauvegardé :", OUTPUT_PATH)

In [ ]:
submission = model_df_test[["airport", "airport_alert_id", "date"]].copy()
submission["prediction_date"] = submission["date"]
submission["predicted_date_end_alert"] = submission["date"]
submission["confidence"] = test_proba_20

submission = submission[
    ["airport", "airport_alert_id", "prediction_date", "predicted_date_end_alert", "confidence"]
].copy()

submission = submission.sort_values(["airport", "airport_alert_id", "prediction_date"]).reset_index(drop=True)

print(submission.head())
print(submission.shape)